# Module 9: Validation

**Task 9**: Validate key observations using visual checks (satellite imagery) and avoid misinterpreting classification noise as real change.

**Deliverables**:
- Validation point table with coordinates and results
- Comparison screenshots (before/after)
- Confusion matrix + accuracy metrics
- Noise/limitations documentation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import (
    STATES, YEARS, LULC_CLASSES, LULC_COLORS,
    PROCESSED_DIR, FIGURES_DIR, VALIDATION_DIR,
)
from scripts.lulc_utils import save_composition_csv
from scripts.visualization import save_figure

print('Imports successful!')

## 9.1: Define Validation Points

Select 15–20 locations showing major transitions. Include:
- Known real changes (urban expansion, highway construction)
- Potential noise (coastal zones, cloud-affected, seasonal crops)
- Stable areas (deep forests, desert)

In [ ]:
# Define validation points
# Update these coordinates based on your actual analysis results

validation_points = [
    # Maharashtra — Known urban expansion
    {'id': 1, 'state': 'Maharashtra', 'location': 'Mumbai suburban expansion',
     'lat': 19.15, 'lon': 72.95, 'expected_change': 'Crops → Built',
     'category': 'Real Change'},
    {'id': 2, 'state': 'Maharashtra', 'location': 'Pune IT corridor',
     'lat': 18.55, 'lon': 73.95, 'expected_change': 'Crops → Built',
     'category': 'Real Change'},
    {'id': 3, 'state': 'Maharashtra', 'location': 'Nagpur metro growth',
     'lat': 21.15, 'lon': 79.10, 'expected_change': 'Crops → Built',
     'category': 'Real Change'},
    {'id': 4, 'state': 'Maharashtra', 'location': 'Mumbai-Pune Expressway',
     'lat': 18.80, 'lon': 73.40, 'expected_change': 'Shrub → Built',
     'category': 'Real Change'},
    
    # Maharashtra — Potential noise
    {'id': 5, 'state': 'Maharashtra', 'location': 'Konkan coast',
     'lat': 17.00, 'lon': 73.30, 'expected_change': 'Water edge fluctuation',
     'category': 'Potential Noise'},
    {'id': 6, 'state': 'Maharashtra', 'location': 'Vidarbha cropland',
     'lat': 20.50, 'lon': 78.50, 'expected_change': 'Crops ↔ Bare (seasonal)',
     'category': 'Potential Noise'},
    {'id': 7, 'state': 'Maharashtra', 'location': 'Western Ghats forest edge',
     'lat': 17.50, 'lon': 73.80, 'expected_change': 'Cloud/shadow artifacts',
     'category': 'Potential Noise'},
    
    # Maharashtra — Stable areas
    {'id': 8, 'state': 'Maharashtra', 'location': 'Sahyadri dense forest',
     'lat': 17.80, 'lon': 73.60, 'expected_change': 'Stable Trees',
     'category': 'Stable'},
    {'id': 9, 'state': 'Maharashtra', 'location': 'Jayakwadi reservoir',
     'lat': 19.50, 'lon': 75.40, 'expected_change': 'Stable Water',
     'category': 'Stable'},
    
    # Maharashtra — Water body changes
    {'id': 10, 'state': 'Maharashtra', 'location': 'Ujani Dam area',
     'lat': 18.05, 'lon': 75.10, 'expected_change': 'Water level change',
     'category': 'Real Change'},
    
    # Sikkim
    {'id': 11, 'state': 'Sikkim', 'location': 'Gangtok expansion',
     'lat': 27.34, 'lon': 88.61, 'expected_change': 'Trees → Built',
     'category': 'Real Change'},
    {'id': 12, 'state': 'Sikkim', 'location': 'Namchi town growth',
     'lat': 27.17, 'lon': 88.35, 'expected_change': 'Trees → Built',
     'category': 'Real Change'},
    {'id': 13, 'state': 'Sikkim', 'location': 'Teesta river bank',
     'lat': 27.25, 'lon': 88.55, 'expected_change': 'Water edge fluctuation',
     'category': 'Potential Noise'},
    {'id': 14, 'state': 'Sikkim', 'location': 'Kangchenjunga slopes',
     'lat': 27.70, 'lon': 88.15, 'expected_change': 'Snow/Ice ↔ Bare (seasonal)',
     'category': 'Potential Noise'},
    {'id': 15, 'state': 'Sikkim', 'location': 'Kanchenjunga NP forest',
     'lat': 27.60, 'lon': 88.20, 'expected_change': 'Stable Trees',
     'category': 'Stable'},
]

val_df = pd.DataFrame(validation_points)
print(f'{len(val_df)} validation points defined')
display(val_df)

## 9.2: Visual Comparison using GEE

Use `geemap` to create side-by-side Sentinel-2 true-color composites.

In [ ]:
try:
    import ee
    import geemap
    
    ee.Initialize()
    
    def create_validation_map(point, years=[2016, 2020, 2025]):
        """Create a geemap comparison for a validation point."""
        lat, lon = point['lat'], point['lon']
        
        Map = geemap.Map(center=[lat, lon], zoom=13)
        
        for year in years:
            # Sentinel-2 true color composite
            s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
                .filterDate(f'{year}-01-01', f'{year}-12-31') \
                .filterBounds(ee.Geometry.Point(lon, lat)) \
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
                .median() \
                .divide(10000)
            
            vis = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}
            Map.addLayer(s2, vis, f'Sentinel-2 {year}')
            
            # Dynamic World LULC
            dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
                .filterDate(f'{year}-01-01', f'{year}-12-31') \
                .filterBounds(ee.Geometry.Point(lon, lat)) \
                .select('label').mode()
            
            dw_vis = {'min': 0, 'max': 8,
                      'palette': list(LULC_COLORS.values())}
            Map.addLayer(dw, dw_vis, f'LULC {year}')
        
        Map.addLayerControl()
        return Map
    
    # Create maps for key validation points
    for _, point in val_df.iterrows():
        if point['category'] == 'Real Change':
            print(f'Creating map for: {point["location"]}...')
            m = create_validation_map(point)
            # Save as HTML
            html_path = VALIDATION_DIR / f'map_{point["id"]}_{point["location"].replace(" ","_")}.html'
            m.to_html(str(html_path))
            print(f'  Saved: {html_path}')

except ImportError:
    print('geemap not available — use Google Earth Pro or dynamicworld.app for visual validation')
except Exception as e:
    print(f'GEE error: {e}')
    print('Use Google Earth Pro or dynamicworld.app for visual validation')

## 9.3: Noise Identification & Documentation

In [ ]:
# Document known classification noise patterns
noise_patterns = [
    {
        'type': 'Seasonal Variation',
        'description': 'Cropland ↔ Bare Ground during off-season (Rabi/Kharif cycles)',
        'affected_classes': 'Crops, Bare Ground',
        'mitigation': 'Mode composite across full year reduces this, but some residual noise remains',
        'severity': 'Moderate',
    },
    {
        'type': 'Cloud/Shadow',
        'description': 'Cloudy scenes cause misclassification, especially in monsoon season',
        'affected_classes': 'All classes, especially Water (shadows ≈ water)',
        'mitigation': 'Mode composite helps. Western Ghats most affected.',
        'severity': 'Low-Moderate',
    },
    {
        'type': 'Water Edge Fluctuation',
        'description': 'Reservoir levels vary seasonally, causing Water ↔ Bare Ground transitions',
        'affected_classes': 'Water, Bare Ground',
        'mitigation': 'Focus on core water area changes, not edges',
        'severity': 'Low',
    },
    {
        'type': 'Snow/Ice Seasonality',
        'description': 'Sikkim high elevations show Snow ↔ Bare Ground seasonal changes',
        'affected_classes': 'Snow & Ice, Bare Ground',
        'mitigation': 'Expected for Sikkim — not misclassification, but seasonal dynamics',
        'severity': 'Low (Sikkim only)',
    },
    {
        'type': 'Grass/Shrub Confusion',
        'description': 'Dynamic World sometimes confuses Grass and Shrub & Scrub',
        'affected_classes': 'Grass, Shrub & Scrub',
        'mitigation': 'Aggregate Grass + Shrub for vegetation analysis (done in Module 8)',
        'severity': 'Low',
    },
]

noise_df = pd.DataFrame(noise_patterns)
print('Known Classification Noise Patterns:')
display(noise_df)

# Save
save_composition_csv(noise_df, VALIDATION_DIR / 'noise_patterns.csv')

## 9.4: Mini Accuracy Assessment

Create a confusion matrix from ~100 random validation points.

In [ ]:
# Template for accuracy assessment
# Fill in after visual interpretation of random points

# Example confusion matrix (replace with actual values after validation)
# Rows = classified (Dynamic World), Columns = reference (visual interpretation)

class_names_short = ['Water', 'Trees', 'Grass', 'Flooded', 'Crops',
                     'Shrub', 'Built', 'Bare', 'Snow']

# Placeholder confusion matrix — UPDATE WITH REAL VALUES
confusion_matrix = np.array([
    [12, 0, 0, 1, 0, 0, 0, 0, 0],  # Water
    [0, 15, 1, 0, 0, 1, 0, 0, 0],   # Trees
    [0, 1, 8, 0, 1, 2, 0, 0, 0],    # Grass
    [1, 0, 0, 3, 0, 0, 0, 0, 0],    # Flooded
    [0, 0, 1, 0, 18, 0, 1, 1, 0],   # Crops
    [0, 1, 2, 0, 0, 7, 0, 0, 0],    # Shrub
    [0, 0, 0, 0, 1, 0, 14, 0, 0],   # Built
    [0, 0, 0, 0, 1, 0, 0, 6, 0],    # Bare
    [0, 0, 0, 0, 0, 0, 0, 0, 4],    # Snow
])

# Overall accuracy
overall_acc = np.trace(confusion_matrix) / confusion_matrix.sum() * 100
print(f'Overall Accuracy: {overall_acc:.1f}%')

# Per-class user and producer accuracy
print(f'\n{"Class":15s} {"User Acc":>10s} {"Producer Acc":>12s}')
print('-' * 40)
for i, name in enumerate(class_names_short):
    row_sum = confusion_matrix[i, :].sum()
    col_sum = confusion_matrix[:, i].sum()
    user_acc = (confusion_matrix[i, i] / row_sum * 100) if row_sum > 0 else 0
    prod_acc = (confusion_matrix[i, i] / col_sum * 100) if col_sum > 0 else 0
    print(f'{name:15s} {user_acc:9.1f}% {prod_acc:11.1f}%')

# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names_short, yticklabels=class_names_short, ax=ax)
ax.set_xlabel('Reference (Visual)')
ax.set_ylabel('Classified (Dynamic World)')
ax.set_title(f'Confusion Matrix — Overall Accuracy: {overall_acc:.1f}%')
plt.tight_layout()
save_figure(fig, 'confusion_matrix.png', 'lulc_maps')
plt.show()

## 9.5: Validation Results Table

In [ ]:
# Add validation results column — fill in after visual checks
val_df['visual_observation'] = '(Fill after visual check)'
val_df['confirmed'] = '(Y/N)'
val_df['notes'] = ''

# Save template
val_df.to_csv(VALIDATION_DIR / 'validation_table.csv', index=False)
print(f'💾 Saved validation template: {VALIDATION_DIR / "validation_table.csv"}')
print('→ Fill in visual_observation and confirmed columns after checking satellite imagery')

display(val_df)
print('\n✅ Module 9 complete.')

---
## Summary
- ✅ 15 validation points defined (real changes, noise, stable)
- ✅ Interactive comparison maps (geemap/HTML)
- ✅ Noise identification documented (5 known patterns)
- ✅ Confusion matrix template with accuracy metrics
- ✅ Validation results table template

**Action required**: Complete the visual checks using Google Earth / Dynamic World Explorer and update the validation table.

**Next**: `10_extra_analysis.ipynb`